In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_contents

load_dotenv(override=True)
from IPython.display import Markdown, display
openai = OpenAI(base_url='http://localhost:11434/v1',api_key='ollama')
system_message = "You are a helpful assistant"
def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model='llama3.2:1b', messages=messages)
    return response.choices[0].message.content


system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""


openai = OpenAI(base_url='http://localhost:11434/v1',api_key='ollama')
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='llama3.2:1b',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

# def stream_claude(prompt):
#     messages = [
#         {"role": "system", "content": system_message},
#         {"role": "user", "content": prompt}
#       ]
#     stream = anthropic.chat.completions.create(
#         model='claude-sonnet-4-5-20250929',
#         messages=messages,
#         stream=True
#     )
#     result = ""
#     for chunk in stream:
#         result += chunk.choices[0].delta.content or ""
#         yield result


def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)

    if model=="llama3.2:1b":
        result = stream_gpt(prompt)
    # elif model=="Claude":
    #     result = stream_claude(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result



name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["llama3.2:1b", "llama3.2:1b"], label="Select model", value="llama3.2:1b")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
            ["welocalize", "https://www.welocalize.com/", "llama3.2:1b"],
            ["Edward Donner", "https://edwarddonner.com", "llama3.2:1b"]
        ],
    flagging_mode="never"
    )
view.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
